# Environment

## Action Space
4 Discrete actions
- Up
- Down
- Left
- Right

## Observation Space
- Board Size (Variable)
- Valid Moves (4 cardinal directions)
- Score (Cumulation of combinations' value)
- Highest Value (Highest index)

## Episode Length
- 100 games of 2048?

## Default Reward
- ✓ Cumulation of all combinations per move
- Standard reward
    - r=merged tile value

- Log reward
    - r=log2(merged tile)

- Number-of-merges reward
- r=#merges

- Survival reward
    - +1 every valid move

- Game-over penalty
    - -1 when game ends

- Maximum-tile reward
    - r= Δlog2(max tile)

## Preprocessing
- Seed per game
- Instead of Factors of 2, its linear, no more log

## Processing
- Discount Function(gamma): 0.99 (Long term reward)



## Imports

In [1]:
import math
import random, collections
import matplotlib.pyplot as plt
import torch, torch.nn as nn, torch.optim as optim
from Resources import Env2048
import numpy as np

## Hyperparameters

In [2]:
TRAIN = True
RENDER = not TRAIN

NUM_AGENTS = 4
NUM_EPISODES = 100
MAX_STEPS = 500
BATCH_SIZE = 64
DISCOUNT = 0.99
LEARNING_RATE = 1e-4 # TODO: Why this value?
# LR_DECAY = 0.001
BUFFER_SIZE = 50000
MIN_REPLAY_SIZE = 5000
EPS_START = 1.0
EPS_END = 0.05
EPS_DECAY = 0.995
TAU = 0.001 # TODO: Why this value?

## Seed

In [3]:
SEED = 21
random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)

## Environment

In [4]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
env = Env2048.Env2048(seed=SEED) # TODO: Add render capabilities

## Normalize State

In [5]:
def preprocessState(board):
    return np.array([
        [
            0 if val == 0 else math.log2(val)
            for val in row
        ]
        for row in board
    ], dtype=np.float32).flatten()

## RL Algorithm
Cant use just a Q-table since the state is large, therefore a nn can to be used/ DQN

In [ ]:
# input size  = state representation size
# output size = number of possible actions

class QNet(nn.Module):
    # 4 Discrete moves → 4 outputs/ Q-values
    # Exploration is E greedy & argmax therefore
    def __init__(self, state_dim, action_dim):
        neuronCount = state_dim * 16
        super().__init__()

        self.network = nn.Sequential(
            # nn.Linear(state_dim, neuronCount), # fully connected layer
            # nn.ReLU(), # Activation Function
            # nn.Linear(neuronCount, neuronCount),
            # nn.ReLU(),
            # nn.Linear(neuronCount, action_dim) # 4 output values
            nn.Linear(16, 128), # fully connected layer
            nn.ReLU(), # Activation Function
            nn.Linear(128, 128),
            nn.ReLU(),
            nn.Linear(128, 4) # 4 output values
        )

    def forward(self, state):
        return self.network(state)

## Replay Buffer
Instead of immediately learning from every experience, store experiences:
$$(s,a,r,s',done)$$

In [7]:
class ReplayBuffer:
    def __init__(self, capacity):
        self.buffer = collections.deque(maxlen=capacity)

    def push(self, state, action, reward, next_state, done):
        self.buffer.append((state, action, reward, next_state, done))

    def sample(self, batch_size):
        indices = np.random.choice(len(self.buffer), batch_size, replace=False)
        batch = [self.buffer[i] for i in indices]
        state, action , reward, next_state, done = zip(*batch)
        return (
            torch.tensor(np.stack(state), dtype=torch.float32, device=DEVICE),
            torch.tensor(action, dtype=torch.long, device=DEVICE),
            torch.tensor(reward, dtype=torch.float32, device=DEVICE),
            torch.tensor(np.stack(next_state), dtype=torch.float32, device=DEVICE),
            torch.tensor(done, dtype=torch.float32, device=DEVICE),
        )

    def __len__(self):
        return len(self.buffer)

## Agent
The agent is the player. The player should only be able to:
- Move:
    - Up, down, left, right
- Observe:
    - Board/ State
    - Reward
    

In [ ]:
class RLAgent():
    def __init__(self, env, replay_buffer):
        self.env = env

        obs_dim = env.observation_space.shape[0]
        n_actions = env.action_space.n

        self.online = QNet(obs_dim, n_actions).to(DEVICE)
        self.target = QNet(obs_dim, n_actions).to(DEVICE)
        self.target.load_state_dict(self.online.state_dict())
        self.optimizer = optim.Adam(
            self.online.parameters(),
            lr=LEARNING_RATE,
            # weight_decay=LR_DECAY
        )
        
        self.replay = replay_buffer
        self.steps = 0
        
        self.validActions = env.getValidActions()

    def chooseAction(self, state, epsilon):
        if random.random() < epsilon:
            return self.env.action_space.sample()

        with torch.no_grad():
            state = torch.tensor(
                state,
                dtype=torch.float32,
                device=DEVICE
            ).flatten()

            q_values = self.online(state)

        return torch.argmax(q_values).item()

    def soft_update(self):
        for target_net, online_net in zip(self.target.parameters(), self.online.parameters()):
            target_net.data.copy_(online_net.data * TAU + target_net.data * (1 - TAU))

    def learn(self, batch_size):
        states, actions, rewards, next_states, dones = self.replay.sample(batch_size)
        actions_idx = actions.unsqueeze(1)          
        rewards = rewards.unsqueeze(1)              
        dones = dones.unsqueeze(1)                  

        q_values = self.online(states).gather(1, actions_idx) 

        # Double DQN target
        with torch.no_grad():
            next_actions = self.online(next_states).argmax(dim=1, keepdim=True) 
            next_q = self.target(next_states).gather(1, next_actions)  
            # Bellman Equation
            target_q = rewards +  DISCOUNT * (1.0 - dones) * next_q              

        loss = nn.SmoothL1Loss()(q_values, target_q)

        # Backprop
        self.optimizer.zero_grad()
        # loss.backward()
        torch.nn.utils.clip_grad_norm_(self.online.parameters(), 5)
        self.optimizer.step()

        self.steps += 1
        self.soft_update()
        return loss.item()

## Init

In [9]:
replay_buffer = ReplayBuffer(BUFFER_SIZE)
agent = RLAgent(env, replay_buffer)

## Random Buffer values
The buffer is pre-filled with random actions to stabilize the gradients and pushed to the shared replay buffer used by all the agents.

In [10]:
def buffer_prefill():
    env.reset(SEED)
    state = preprocessState(env.board)
    for _ in range(MIN_REPLAY_SIZE):
        action = env.action_space.sample()
        oldBoard, newState, terminated, truncated, reward, validActions, info = env.step(action)
        replay_buffer.push(state, action, reward, newState, terminated)
        if terminated:
            env.reset(SEED)
            state = preprocessState(env.board)
        else:
            state = preprocessState(newState)

## Plots

In [11]:
def plot_training(rewards, losses, average=50, max_reward=1000):
    rewards = np.array(rewards)
    rewards_history = np.clip(rewards, None, max_reward)

    if len(rewards) >= average:
        sma = np.clip(
            np.convolve(rewards, np.ones(average) / average, mode="valid"),
            None,
            max_reward
        )
    else:
        sma = None

    plt.figure()
    plt.title(f"Rewards")
    plt.plot(rewards_history, label="Raw Reward")
    if sma is not None:
        plt.plot(sma, label=f"SMA {average}")
    plt.xlabel("Episode")
    plt.ylabel("Rewards")
    plt.legend()
    # plt.savefig("Results/reward_plot.png", dpi=600, bbox_inches="tight")
    plt.show()

    plt.figure()
    plt.title(f"Loss")
    plt.plot(losses, label="Loss")
    plt.xlabel("Training Step")
    plt.ylabel("Loss")
    # plt.savefig("Results/loss_plot.png", dpi=600, bbox_inches="tight")
    plt.show()

## Train

In [ ]:
def train():
    epsilon = EPS_START
    rewards_history = []
    loss_history = []

    for episode in range(NUM_EPISODES):
        env.reset(SEED)
        state = env.board
        
        ep_reward = 0.0
        ep_steps = 0
        
        epsilon = max(EPS_END, epsilon * EPS_DECAY)

        for step in range(MAX_STEPS):
            # get action
            action = agent.chooseAction(state, epsilon)
            print("Action", action)
            # use action
            oldBoard, newState, terminated, truncated, reward, validActions, info = env.step(action)
            # print(
            #     "Action:",
            #     action,
            #     "| Reward:",
            #     reward,
            #     "| Valid moves:",
            #     validActions,
            #     "| Done:",
            #     terminated or truncated,
            # )
            
            # Replay buffer
            replay_buffer.push(
                state,
                action,
                reward,
                newState,
                terminated or truncated
            )

            state = newState
            ep_reward += reward
            ep_steps += 1

            # 4 is learning delay
            if step % 4 == 0 and len(replay_buffer) >= BATCH_SIZE:
                # for agent in agents:
                loss = agent.learn(BATCH_SIZE)
                loss_history.append(loss)

            if terminated:
                break
            
            rewards_history.append(ep_reward)

            if ep_steps < 200 or episode % 10 == 0:
            # if ep_steps < 200 or episode % 100 == 0:
                print(
                    f"Episode {episode:4d} | "
                    f"Reward {ep_reward:7.2f} | "
                    f"Steps {ep_steps:3d} | "
                    f"Epsilon {epsilon:.3f} | "
                )
                
        torch.save(agent.online.state_dict(), f"Results/dqn_2048{NUM_EPISODES}.pth")
        print("Model saved.")
        plot_training(rewards_history, loss_history)


## Test

In [13]:
def test(episodes):
    for episode in range(episodes):
        env.reset(seed=SEED)
        state = env.board 
        episode_reward = 0.0
        ep_steps = 0

        for _ in range(MAX_STEPS):
            action = agent.chooseAction(state, epsilon=0.0)
            oldBoard, newState, terminated, truncated, reward, validActions, info = env.step(action)

            state = newState
            episode_reward += reward
            ep_steps += 1
            if terminated:
                break
        print(
            f"Episode {episode+1:3d} | "
            f"Steps {ep_steps:3d} | "
            f"Return {episode_reward:7.2f}"
        )

## Main Runner

In [14]:
if __name__ == "__main__":
    if TRAIN:
        buffer_prefill()
        train()
    else:
        # for agent in agents:
        agent.online.load_state_dict(torch.load(f"Results/dqn_mountaincar_{NUM_EPISODES}.pth", map_location=DEVICE))
        agent.target.load_state_dict(agent.online.state_dict())
        test(2)

Action 0
Episode    0 | Reward    0.00 | Steps   1 | Epsilon 0.995 | 
Action 3
Episode    0 | Reward    8.00 | Steps   2 | Epsilon 0.995 | 
Action 2
Episode    0 | Reward    8.00 | Steps   3 | Epsilon 0.995 | 
Action 0
Episode    0 | Reward   12.00 | Steps   4 | Epsilon 0.995 | 
Action 0
Episode    0 | Reward   12.00 | Steps   5 | Epsilon 0.995 | 
Action 1
Episode    0 | Reward   12.00 | Steps   6 | Epsilon 0.995 | 
Action 1
Model saved.
Action 3
Episode    1 | Reward    0.00 | Steps   1 | Epsilon 0.990 | 
Action 1
Model saved.
Action 2
Episode    2 | Reward    0.00 | Steps   1 | Epsilon 0.985 | 
Action 3
Episode    2 | Reward    0.00 | Steps   2 | Epsilon 0.985 | 
Action 1
Episode    2 | Reward    0.00 | Steps   3 | Epsilon 0.985 | 
Action 1
Episode    2 | Reward    4.00 | Steps   4 | Epsilon 0.985 | 
Action 3
Episode    2 | Reward   16.00 | Steps   5 | Epsilon 0.985 | 
Action 0
Episode    2 | Reward   24.00 | Steps   6 | Epsilon 0.985 | 
Action 1
Episode    2 | Reward   40.00 | Steps